## Imports

In [1]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Configuration

In [2]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,        # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json")
palette

Project not installed in the current env, activate the correct env or install it with:
	pip install -e .


{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [3]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [4]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [5]:
tags = [
    "TSVM"
]  

In [6]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'TSVM'}}]}
There are 9 runs respecting these conditions.


In [ ]:
models = ['ViT-B-32', 'ViT-B-16', 'ViT-L-14']

In [8]:
ref_run = runs[0]

In [9]:
print(set(ref_run.history().columns))

{'loss/test/Flowers102', 'acc/test/Cars', '_timestamp', 'normalized_acc/test/Flowers102', 'loss/test/RESISC45', 'loss/test/STL10', 'acc/test/DTD', 'normalized_acc/test/CIFAR100', 'loss/test/DTD', 'acc/test/avg', 'loss/test/OxfordIIITPet', '_runtime', 'loss/test/GTSRB', 'normalized_acc/test/SVHN', 'acc/test/Flowers102', 'acc/test/SUN397', 'acc/test/RESISC45', 'normalized_acc/test/SUN397', 'acc/test/SVHN', 'loss/test/Cars', '_step', 'loss/test/CIFAR100', 'radar', 'loss/test/SUN397', 'loss/test/EuroSAT', 'acc/test/PCAM', 'normalized_acc/test/DTD', 'acc/test/MNIST', 'loss/test/FER2013', 'normalized_acc/test/OxfordIIITPet', 'acc/test/STL10', 'normalized_acc/test/MNIST', 'epoch', 'normalized_acc/test/RESISC45', 'normalized_acc/test/Cars', 'normalized_acc/test/avg', 'normalized_acc/test/GTSRB', 'normalized_acc/test/STL10', 'acc/test/CIFAR100', 'acc/test/OxfordIIITPet', 'normalized_acc/test/FER2013', 'acc/test/EuroSAT', 'acc/test/GTSRB', 'trainer/global_step', 'normalized_acc/test/EuroSAT', 'l

In [10]:
print(ref_run.config['core/tags'])

['TSVM', 'benchmark', 'static_merge', 'n14', 'ViT-B-16']


#### Hparams

In [11]:
benchmarks = ['n8', 'n14', 'n20']
models = ['ViT-B-32', 'ViT-B-16', 'ViT-L-14']

In [18]:
avg_accs = {model: {benchmark: {'avg_acc': 0.0, 'norm_acc': 0.0} for benchmark in benchmarks} for model in models}

for run in runs:
    model = run.config['nn/encoder/model_name']

    try:
        N = run.config['num_tasks']
    except KeyError:
        N = run.config['ntasks']

    benchmark = f'n{N}'

    avg_accs[model][benchmark]['avg_acc'] = run.summary['acc/test/avg']
    avg_accs[model][benchmark]['norm_acc'] = run.summary['normalized_acc/test/avg']

In [20]:
avg_accs

{'ViT-B-32': {'n8': {'avg_acc': 0.8321913778781891,
   'norm_acc': 0.917881704866886},
  'n14': {'avg_acc': 0.7861627851213727, 'norm_acc': 0.8799804278782436},
  'n20': {'avg_acc': 0.7555608719587326, 'norm_acc': 0.8430375337600708}},
 'ViT-B-16': {'n8': {'avg_acc': 0.8551733419299126,
   'norm_acc': 0.9219112694263458},
  'n14': {'avg_acc': 0.8138026084218707, 'norm_acc': 0.8875468926770347},
  'n20': {'avg_acc': 0.7880484014749527, 'norm_acc': 0.8551690846681594}},
 'ViT-L-14': {'n8': {'avg_acc': 0.9124660044908524,
   'norm_acc': 0.9671125411987304},
  'n14': {'avg_acc': 0.8877613927636828, 'norm_acc': 0.94869179385049},
  'n20': {'avg_acc': 0.8748595148324967, 'norm_acc': 0.929747748374939}}}

In [ ]:
# print latex

row = ''
for model in models:

    for benchmark in benchmarks:
        avg_acc = avg_accs[model][benchmark]['avg_acc']
        norm_acc = avg_accs[model][benchmark]['norm_acc']

        row += "{avg_acc:.2f} & {norm_acc:.2f} & ", end='')
    print("\\\\")
